# Autoresearch — analyse des expériences (Stage 1)

Analyse des résultats de la boucle autonome à partir de `results.tsv` (métrique : `val_dice`, **plus haut = meilleur**).

In [ ]:
import csv
import itertools
import pathlib

import matplotlib.pyplot as plt
import numpy as np

HERE = pathlib.Path.cwd()
if (HERE / "autoresearch").is_dir():
    HERE = HERE / "autoresearch"  # notebook lancé depuis la racine du repo

# TSV : 5 colonnes commit, val_dice, seconds, status, description
with open(HERE / "results.tsv", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))
for r in rows:
    r["val_dice"] = float(r["val_dice"])
    r["seconds"] = float(r["seconds"])
    r["status"] = r["status"].strip().upper()
    r["description"] = r["description"].strip()

print(f"Total d'expériences : {len(rows)}")
for r in rows[:10]:
    print(
        f"  {r['commit']}  {r['val_dice']:.6f}  {r['seconds']:6.1f}s  {r['status']:8s} {r['description']}"
    )

In [ ]:
statuses = [r["status"] for r in rows]
n_keep, n_discard, n_crash = (statuses.count(s) for s in ("KEEP", "DISCARD", "CRASH"))
print("Issues des expériences :")
print(f"  KEEP     {n_keep}\n  DISCARD  {n_discard}\n  CRASH    {n_crash}")
if n_keep + n_discard:
    print(f"\nTaux de keep : {n_keep}/{n_keep + n_discard} = {n_keep / (n_keep + n_discard):.1%}")

In [ ]:
# Toutes les expériences GARDÉES (les améliorations qui ont tenu)
print(f"Expériences gardées ({n_keep}) :\n")
for i, r in enumerate(rows):
    if r["status"] == "KEEP":
        print(f"  #{i:3d}  dice={r['val_dice']:.6f}  {r['seconds']:6.1f}s  {r['description']}")

## Val Dice au fil des expériences

Évolution du meilleur `val_dice` gardé. Le *running max* en escalier est la « frontière » : le meilleur résultat atteint jusque-là.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# On retire les crashs du tracé
valid = [r for r in rows if r["status"] != "CRASH"]
x = np.arange(len(valid))
dice_v = np.array([r["val_dice"] for r in valid])
is_keep = np.array([r["status"] == "KEEP" for r in valid])

baseline = dice_v[0]
# Ne tracer que la zone intéressante : au-dessus (ou presque) de la baseline
shown = dice_v >= baseline - 0.0005

# Expériences jetées : points gris discrets en fond
disc = shown & ~is_keep
ax.scatter(x[disc], dice_v[disc], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Jetées")

# Expériences gardées : points verts bien visibles
ax.scatter(
    x[is_keep],
    dice_v[is_keep],
    c="#2ecc71",
    s=50,
    zorder=4,
    label="Gardées",
    edgecolors="black",
    linewidths=0.5,
)

# Meilleur courant (running max) en escalier
kept_idx = x[is_keep]
kept_dice = dice_v[is_keep]
running_best = np.maximum.accumulate(kept_dice)
ax.step(
    kept_idx,
    running_best,
    where="post",
    color="#27ae60",
    linewidth=2,
    alpha=0.7,
    zorder=3,
    label="Meilleur courant",
)

# Annoter chaque expérience gardée avec sa description
for idx, d in zip(kept_idx, kept_dice):
    desc = valid[idx]["description"]
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(
        desc,
        (idx, d),
        textcoords="offset points",
        xytext=(6, -6),
        fontsize=8.0,
        color="#1a7a3a",
        alpha=0.9,
        rotation=-30,
        ha="left",
        va="top",
    )

ax.set_xlabel("Expérience #", fontsize=12)
ax.set_ylabel("Val Dice (plus haut = meilleur)", fontsize=12)
ax.set_title(
    f"Autoresearch Stage 1 : {len(rows)} expériences, {n_keep} améliorations gardées", fontsize=14
)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Axe Y : de juste sous la baseline à juste au-dessus du meilleur
best = kept_dice.max()
margin = max((best - baseline) * 0.15, 0.002)
ax.set_ylim(baseline - margin, best + margin)

plt.tight_layout()
plt.savefig(HERE / "progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé dans", HERE / "progress.png")

## Statistiques résumées

In [ ]:
kept = [(i, r) for i, r in enumerate(rows) if r["status"] == "KEEP"]
baseline = rows[0]["val_dice"]
best_i, best_row = max(kept, key=lambda t: t[1]["val_dice"])

print(f"Baseline val_dice : {baseline:.6f}")
print(f"Meilleur val_dice : {best_row['val_dice']:.6f}")
gain = best_row["val_dice"] - baseline
print(f"Gain total        : {gain:+.6f} ({gain / baseline * 100:+.2f} %)")
print(f"Meilleure exp.    : #{best_i} {best_row['description']}")
print("\nEffort cumulé par amélioration :")
for i, r in kept:
    print(f"  Expérience #{i:3d} : dice={r['val_dice']:.6f}  {r['description']}")

## Top des gains (expériences gardées triées par amélioration)

In [ ]:
# Le delta de chaque keep est mesuré par rapport au keep précédent
# (les expériences sont cumulatives : chacune part du dernier état gardé)
hits = []
for (_, prev), (_, r) in itertools.pairwise(kept):
    hits.append((r["val_dice"] - prev["val_dice"], r))
hits.sort(key=lambda t: t[0], reverse=True)

print(f"{'Rang':>4}  {'Delta':>9}  {'Dice':>9}  Description")
print("-" * 80)
for rank, (delta, r) in enumerate(hits, 1):
    print(f"{rank:4d}  {delta:+.6f}  {r['val_dice']:.6f}  {r['description']}")
print(f"\n{'':>4}  {sum(d for d, _ in hits):+.6f}  {'':>9}  amélioration TOTALE vs baseline")

## Meilleures et pires images de la val

Image, mask obtenu (`detect.predict`) et mask cible pour les images de la val au meilleur et au pire Dice.

In [ ]:
import sys

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import detect
import prepare

images, masks = prepare._load(prepare.VAL_DIR, prepare.DEFAULT_BATCH_SIZE)
preds = [np.asarray(detect.predict(im.copy())) for im in images]
scores = np.array([prepare.dice(p, t) for p, t in zip(preds, masks)])
val_dice = scores.mean()
print(f"val_dice : {val_dice:.6f}  ({len(scores)} images)")

N_SHOW = 4
order = np.argsort(scores)
groups = {"Meilleures": order[::-1][:N_SHOW], "Pires": order[:N_SHOW]}

for name, idxs in groups.items():
    fig, axes = plt.subplots(len(idxs), 3, figsize=(10, 3.3 * len(idxs)), squeeze=False)
    for row, i in zip(axes, idxs):
        for ax, img, title in zip(
            row,
            (images[i], preds[i], masks[i]),
            (f"#{i} image  dice={scores[i]:.3f}", "mask obtenu", "mask cible"),
        ):
            ax.imshow(img, origin="lower", cmap="gray")
            ax.set_title(title, fontsize=10)
            ax.axis("off")
    fig.suptitle(f"{name} images de la val (val_dice={val_dice:.4f})", fontsize=13)
    plt.tight_layout()
    plt.show()